# Verified installation

**EXPERIMENTAL — KAGGLE DUAL-T4 VALIDATION PENDING.** Not executed by the author; outputs are produced only when *you* run it.

Accelerator: **GPU T4 x2**, Internet **ON** (notebooks 02–06 download the 8 pinned helper wheels, ~57 MB, and 05/06 download the model). Run notebooks in order 01 → 06 (each Kaggle notebook is a fresh session, so notebooks 03–06 re-run the idempotent installer first); each stage writes JSON to `/kaggle/working/stokastisk-acceptance/`.

Verifies SHA256SUMS **before** installing; plans from a before-snapshot and aborts untouched on any conflict; installs both wheels `--no-deps`, then only the 8 exact pins in `requirements/kaggle-t4-core.txt` (also `--no-deps`; **no resolver ever runs**). Success: all PASS, every preinstalled package unchanged (torch, Triton, Transformers 5.x, CUDA 13.0 packages), no FlashAttention/FlashInfer, imports + `TRITON_ATTN` class resolve. Re-running changes nothing. Use `--dry-run` to see the plan only.

## 1. Setup — locate the release bundle

In [1]:
import glob, hashlib, json, os, pathlib, shutil, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import hashlib, json, os, pathlib, shutil, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor

HF_REPO = "waqasm86/stokastisk"
HF_BASE = f"https://huggingface.co/{HF_REPO}/resolve"


def _open(url, headers=None):
    return urllib.request.urlopen(urllib.request.Request(url, headers=headers or {"User-Agent": "stokastisk-fetch/1"}), timeout=60)


def _range_ok(url):
    if not url.startswith("http"):
        return None
    with _open(url, {"Range": "bytes=0-0", "User-Agent": "stokastisk-fetch/1"}) as r:
        cr = r.headers.get("Content-Range")                      # "bytes 0-0/37080766"
        return int(cr.rsplit("/", 1)[1]) if r.status == 206 and cr else None


def _download(url, dest, workers=1, retries=3):
    """Download to `<dest>.part`, check free disk space and the advertised length, then replace `dest` atomically; the partial file is removed on any failure."""
    dest = pathlib.Path(dest); part = dest.with_name(dest.name + ".part")
    part.unlink(missing_ok=True)
    try:
        total = _range_ok(url)
        if total and shutil.disk_usage(dest.parent).free < total + (64 << 20):
            raise RuntimeError(f"not enough free disk space in {dest.parent} for {total / 1e6:.0f} MB")
        _download_to(url, part, workers, retries)
        if total and part.stat().st_size != total:
            raise RuntimeError(f"incomplete download: {part.stat().st_size} of {total} bytes")
        os.replace(part, dest)
    except BaseException:
        part.unlink(missing_ok=True)
        raise


def _download_to(url, dest, workers=1, retries=3):
    dest = pathlib.Path(dest); total = _range_ok(url)
    t0 = time.time()
    if total and total > (4 << 20):
        chunk = -(-total // workers); parts = [(i * chunk, min(total, (i + 1) * chunk) - 1) for i in range(workers) if i * chunk < total]
        with open(dest, "wb") as f:
            f.truncate(total)

        def part(rng):
            a, b = rng
            for attempt in range(retries):
                try:
                    with _open(url, {"Range": f"bytes={a}-{b}", "User-Agent": "stokastisk-fetch/1"}) as r, open(dest, "r+b") as f:
                        f.seek(a); shutil.copyfileobj(r, f, 1 << 20)
                    return
                except Exception:
                    if attempt == retries - 1: raise
                    time.sleep(2 ** attempt)
        with ThreadPoolExecutor(len(parts)) as ex:
            list(ex.map(part, parts))
        mode = f"{len(parts)} parallel ranges"
    else:
        for attempt in range(retries):
            try:
                with _open(url) as r, open(dest, "wb") as f:
                    shutil.copyfileobj(r, f, 1 << 20)
                break
            except Exception:
                if attempt == retries - 1: raise
                time.sleep(2 ** attempt)
        mode = "single stream"
    n = dest.stat().st_size
    print(f"downloaded {n / 1e6:.1f} MB in {time.time() - t0:.1f}s ({n / 1e6 / max(time.time() - t0, 1e-6):.1f} MB/s, {mode})")


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


def fetch_release(release, dest="/tmp/stokastisk-release", base=None, revision="main", workers=1):
    """Return the verified, extracted bundle directory (…/<release>). `base` overrides the Hub URL prefix (tests use file://)."""
    base = (base or HF_BASE).rstrip("/")
    root = f"{base}/{revision}/releases/{release}" if base.startswith("http") else f"{base}/releases/{release}"
    dest = pathlib.Path(dest); dest.mkdir(parents=True, exist_ok=True)
    zname = f"{release}-kaggle-input.zip"; zpath = dest / zname
    if zpath.exists(): zpath.unlink()
    with _open(f"{root}/{zname}.sha256") as r:
        expected = r.read().decode().split()[0]
    _download(f"{root}/{zname}", zpath, workers=workers)
    got = _sha256(zpath)
    if got != expected:
        zpath.unlink(); raise RuntimeError(f"SHA256 mismatch for {zname}: expected {expected}, got {got}")
    out = dest / release
    if out.exists(): shutil.rmtree(out)
    with zipfile.ZipFile(zpath) as z:
        assert z.testzip() is None
        for n in z.namelist():
            if n.startswith("/") or ".." in pathlib.PurePosixPath(n).parts or not n.startswith(release + "/"):
                raise RuntimeError(f"unsafe or foreign path in archive: {n}")
        z.extractall(dest)
    m = json.loads((out / "manifests" / "RELEASE-MANIFEST.json").read_text())
    if m.get("name") != release:
        raise RuntimeError(f"manifest name {m.get('name')!r} != {release!r}")
    print(f"release {release}: SHA256 {got[:16]}... verified, extracted to {out}")
    return out

# Explicit release identity: only the bundle directory named RELEASE whose RELEASE-MANIFEST.json carries the same name is accepted
# (an older rc is never silently selected). Source 1: the attached Kaggle dataset. Source 2 (nothing attached): the Hugging Face CDN.
RELEASE = "stokastisk-0.1.0-kaggle-t4-sm75"
cands = [pathlib.Path(p).parent.parent for p in glob.glob("/kaggle/input/**/manifests/SHA256SUMS", recursive=True)]
if os.environ.get("STOKASTISK_BUNDLE"): cands = [pathlib.Path(os.environ["STOKASTISK_BUNDLE"])]
match = [c for c in cands if c.name == RELEASE]
if not match:
    print(f"{RELEASE} is not attached (attached: {sorted(str(c) for c in cands)}); downloading it from Hugging Face (needs Internet ON) ...")
    match = [fetch_release(RELEASE)]
assert len({(c / "manifests" / "SHA256SUMS").read_bytes() for c in match}) == 1, f"several DIFFERENT copies of {RELEASE}: {match}"
BUNDLE = match[0]
_m = json.loads((BUNDLE / "manifests" / "RELEASE-MANIFEST.json").read_text())
assert _m.get("name") == RELEASE, f"manifest name {_m.get('name')!r} != expected {RELEASE!r}"
SCRIPTS = BUNDLE / "scripts" / "kaggle"
os.environ["STOKASTISK_OUT"] = "/kaggle/working/stokastisk-acceptance"
print("release:", RELEASE, "| bundle:", BUNDLE)

release: stokastisk-0.1.0-kaggle-t4-sm75 | bundle: /kaggle/input/datasets/waqasm86/stokastisk-0-1-0-kaggle-t4-sm75-kaggle-input-2/stokastisk-0.1.0-kaggle-t4-sm75


## 2. Dry run (plan only; nothing is installed). **The installation cell below is blocked unless this passes.**

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, str(SCRIPTS / "02_install.py"), str(BUNDLE), "--dry-run"])
assert r.returncode == 0, "dry run failed: do NOT install; read the output above"
print("dry run OK")

PASS SHA256SUMS verified before any install -- all files match
PASS python 3.13 -- 3.13.16
PASS baseline protected (torch/triton exact, transformers>=5.5.3, no prohibited pkgs) -- {'torch': '2.11.0+cu130', 'triton': '3.6.0', 'transformers': '5.20.2', 'numpy': '1.26.4', 'nvidia-cuda-nvcc': '13.0.88', 'nvidia-nvvm': '13.0.88'}
PASS two Tesla T4 (SM75) -- 2 gpu(s) [(7, 5), (7, 5)]
PASS exactly two wheels -- ['stokastisk-0.1.0-py3-none-any.whl', 'vllm-0.25.1-cp38-abi3-linux_x86_64.whl']
PASS wheels contain no FlashAttention/FlashInfer/FlashMLA files -- none
PASS dependency plan has no conflicts (nothing is installed if this fails) -- {'install': {'apache-tvm-ffi': '0.1.9', 'xgrammar': '0.2.5.post1', 'llguidance': '1.7.6', 'cbor2': '6.2.0.post2', 'msgspec': '0.22.0', 'openai-harmony': '0.0.8', 'partial-json-parser': '0.2.1.1.post7', 'pybase64': '1.5.1'}}
PASS download exact pins (no resolver, --no-deps) -- 8 pins

[02_install_dryrun] OVERALL: PASS
dry run OK


## 3. Install

In [3]:
r = subprocess.run([sys.executable, str(SCRIPTS / "02_install.py"), str(BUNDLE)])
RC = {"02_install": r.returncode}
print(RC)

PASS SHA256SUMS verified before any install -- all files match
PASS python 3.13 -- 3.13.16
PASS baseline protected (torch/triton exact, transformers>=5.5.3, no prohibited pkgs) -- {'torch': '2.11.0+cu130', 'triton': '3.6.0', 'transformers': '5.20.2', 'numpy': '1.26.4', 'nvidia-cuda-nvcc': '13.0.88', 'nvidia-nvvm': '13.0.88'}
PASS two Tesla T4 (SM75) -- 2 gpu(s) [(7, 5), (7, 5)]
PASS exactly two wheels -- ['stokastisk-0.1.0-py3-none-any.whl', 'vllm-0.25.1-cp38-abi3-linux_x86_64.whl']
PASS wheels contain no FlashAttention/FlashInfer/FlashMLA files -- none
PASS dependency plan has no conflicts (nothing is installed if this fails) -- {'install': {'apache-tvm-ffi': '0.1.9', 'xgrammar': '0.2.5.post1', 'llguidance': '1.7.6', 'cbor2': '6.2.0.post2', 'msgspec': '0.22.0', 'openai-harmony': '0.0.8', 'partial-json-parser': '0.2.1.1.post7', 'pybase64': '1.5.1'}}
PASS download exact pins (no resolver, --no-deps) -- 8 pins
PASS install wheels --force-reinstall --no-deps (replaces any other stokastisk

## 4. The installed packages must be byte-identical to the bundle's wheels

In [4]:
import importlib, importlib.metadata as md, zipfile, hashlib
for w in sorted((BUNDLE / "wheels").glob("*.whl")):
    pkg = w.name.split("-")[0]
    mod = importlib.import_module(pkg)
    root = pathlib.Path(mod.__file__).resolve().parent
    z = zipfile.ZipFile(w); bad = []
    for n in z.namelist():
        if n.startswith(pkg + "/") and n.endswith((".py", ".so")):
            f = root.parent / n
            if not f.is_file() or hashlib.sha256(f.read_bytes()).hexdigest() != hashlib.sha256(z.read(n)).hexdigest(): bad.append(n)
    print(f"{pkg} {md.version(pkg)} imported from {root}: {'identical to ' + w.name if not bad else 'MISMATCH ' + str(bad)}")
    assert not bad, f"installed {pkg} differs from the bundle wheel {w.name}; uninstall it and re-run the install cell"

stokastisk 0.1.0 imported from /usr/local/lib/python3.13/dist-packages/stokastisk: identical to stokastisk-0.1.0-py3-none-any.whl
vllm 0.25.1 imported from /usr/local/lib/python3.13/dist-packages/vllm: identical to vllm-0.25.1-cp38-abi3-linux_x86_64.whl


## Package the evidence

Creates `/kaggle/working/02-stokastisk-acceptance.zip` (download it from the Output tab).

In [5]:
import shutil
print(shutil.make_archive("/kaggle/working/02-stokastisk-acceptance", "zip", "/kaggle/working", "stokastisk-acceptance"))

/kaggle/working/02-stokastisk-acceptance.zip


## Verdict — this cell fails the notebook if any stage failed

In [6]:
rcs = RC
assert all(v == 0 for v in rcs.values()), f'FAILED stage(s): {rcs} -- see the logs and JSON in /kaggle/working/stokastisk-acceptance/'
print('ALL STAGES PASSED', rcs)

ALL STAGES PASSED {'02_install': 0}
